In [21]:
# Data exploration LLM_ABSA

In [22]:
import pandas as pd
from src.ML_ABSA import ABSA
from src.evaluator.evaluate_absa import evaluate_absa

# Load dataset once
df = pd.read_csv("../data/Yelp Restaurant Reviews.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
# print("\nSample reviews:")
# print(df['Review Text'].head(10))
df

Dataset shape: (19896, 4)
Columns: ['Yelp URL', 'Rating', 'Date', 'Review Text']


,Yelp URL,Rating,Date,Review Text
0,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,1/22/2022,All I can say is they have very good ice cream...
1,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,6/26/2022,Nice little local place for ice cream.My favor...
2,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,8/7/2021,A delicious treat on a hot day! Staff was very...
3,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,7/28/2016,This was great service and a fun crew! I got t...
4,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,6/23/2015,This is one of my favorite places to get ice c...
...,...,...,...,...
19891,https://www.yelp.com/biz/la-pasticceria-las-vegas,4,7/17/2021,Had the chocolate cannoli! The filling was ric...
19892,https://www.yelp.com/biz/la-pasticceria-las-vegas,4,10/21/2019,Love apricot croissant! I bought it at 4:00 PM...
19893,https://www.yelp.com/biz/la-pasticceria-las-vegas,4,10/12/2019,Line was about 25 people long. It went fast! T...
19894,https://www.yelp.com/biz/la-pasticceria-las-vegas,5,4/11/2021,Its hard not to order everything when I come h...


In [23]:
# Initialize analyzer
import os
# In Jupyter notebooks, we need to use a path relative to the notebook location
# Use the local checkpoint copy in the notebooks directory to avoid path issues
checkpoint_path = os.path.join("checkpoints", "ATEPC_MULTILINGUAL_CHECKPOINT")
# Still need the project root for the dataset path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
dataset_path = os.path.join(PROJECT_ROOT, "src", "evaluator", "absa_test.json")

absa = ABSA(model_name=checkpoint_path, device='cpu')
evaluate_absa(absa, dataset_path)


# Run analyzer on 5 random reviews
sample_reviews = df['Review Text'].sample(5)
sample_reviews

Loading PyABSA model from 'checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT' on device: cpu
[2025-10-24 10:54:59] (2.4.2) Load aspect extractor from checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT
[2025-10-24 10:54:59] (2.4.2) config: checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT\fast_lcf_atepc.config
[2025-10-24 10:54:59] (2.4.2) state_dict: checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT\fast_lcf_atepc.state_dict
[2025-10-24 10:54:59] (2.4.2) model: None
[2025-10-24 10:54:59] (2.4.2) tokenizer: checkpoints\ATEPC_MULTILINGUAL_CHECKPOINT\fast_lcf_atepc.tokenizer
[2025-10-24 10:55:00] (2.4.2) Set Model Device: cpu
[2025-10-24 10:55:00] (2.4.2) Device Name: Unknown


C:\Users\phili\venvs\recommenderSystemsProject\Lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
C:\Users\phili\venvs\recommenderSystemsProject\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\phili\venvs\recommenderSystemsProject\Lib\site-packages\pyabsa\tasks\AspectTermExtraction\prediction\aspect_extractor.py:593: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider

Precision: 0.734
Recall:    0.622
F1-score:  0.674


9568     I've been going to 85 C for a few years when I...
9057     I've had this place bookmarked for quite some ...
4973     Mmmm you can't go wrong with sweets, definatel...
10493    Best donuts hands down but get there early or ...
14656    Jarlings has the best ice cream to eat on camp...
Name: Review Text, dtype: object

In [24]:
all_results = []
for i, review in enumerate(sample_reviews, 1):
    results = absa.analyze(str(review))
    print(f"\nRandom Review {i}: {review}")
    for r in results:
        print("   ", r)

    all_results.append({
        "review": review,
        "aspects": [r.aspect for r in results],
        "sentiments": [r.sentiment for r in results]
    })


results_df = pd.DataFrame(all_results)
results_df


C:\Users\phili\venvs\recommenderSystemsProject\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Random Review 1: I've been going to 85 C for a few years when I'd visit CA. Nice that there's one in Vegas. I always find something yummy to calm my sweet tooth. Now it's all grab and go due to Covid-19. No indoor seating or restrooms so don't expect to order a drink and enjoy your pastry in the cafe. Makes me miss hanging out with family enjoying a leisurely chat over some sweets and coffee.
    AspectSentiment(aspect='seating', sentiment='neutral', confidence=0.785, text_span=[2])
    AspectSentiment(aspect='drink', sentiment='neutral', confidence=0.8203, text_span=[13])
    AspectSentiment(aspect='pastry', sentiment='positive', confidence=0.9657, text_span=[17])
    AspectSentiment(aspect='sweets', sentiment='positive', confidence=0.8757, text_span=[13])
    AspectSentiment(aspect='coffee', sentiment='positive', confidence=0.8012, text_span=[15])

Random Review 2: I've had this place bookmarked for quite some time now and I finally made it out there the other day. I put the directi

,review,aspects,sentiments
0,I've been going to 85 C for a few years when I...,"[seating, drink, pastry, sweets, coffee]","[neutral, neutral, positive, positive, positive]"
1,I've had this place bookmarked for quite some ...,"[place, location, parking lot, sign, staff mem...","[positive, negative, negative, positive, posit..."
2,"Mmmm you can't go wrong with sweets, definatel...","[sweets, macaroons, prices]","[positive, negative, positive]"
3,Best donuts hands down but get there early or ...,"[donuts, flavors, donuts, bacon maple]","[positive, positive, positive, positive]"
4,Jarlings has the best ice cream to eat on camp...,"[ice cream, service, employees]","[positive, positive, positive]"
